In [78]:
import pandas as pd
import numpy as np

In [79]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

In [80]:
print(ratings.head())
print("-----------------------------------------------------")
print("-----------------------------------------------------")

print(movies.head())
print("-----------------------------------------------------")
print("-----------------------------------------------------")

print(ratings.info())
print("-----------------------------------------------------")
print("-----------------------------------------------------")

print(movies.info())


   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
-----------------------------------------------------
-----------------------------------------------------
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
-----------------------------------------------------
---------------

In [81]:
# Check
print(ratings.isnull().sum())
print("-----------------------------------------------------")
print("-----------------------------------------------------")
print("-----------------------------------------------------")
print(movies.isnull().sum())

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
-----------------------------------------------------
-----------------------------------------------------
-----------------------------------------------------
movieId    0
title      0
genres     0
dtype: int64


In [82]:
# Handling
ratings.dropna(inplace=True)
movies.dropna(inplace=True)

In [83]:
print(ratings.duplicated().sum())
print("-------------------------------------------------------")
print("-------------------------------------------------------")
print(movies.duplicated().sum())


0
-------------------------------------------------------
-------------------------------------------------------
0


In [84]:
set(ratings['movieId']).issubset(set(movies['movieId']))

True

In [85]:
df = pd.merge(ratings, movies, on="movieId")

In [86]:
print(df.head())
print("------------------------------------------------------")
print(df.isnull().sum())

   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  
------------------------------------------------------
userId       0
movieId      0
rating       0
timestamp    0
title        0
genres       0
dtype: int64


In [87]:
print(df.shape)
print(df.head())

(100836, 6)
   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  


In [88]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [89]:
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)
print(movies.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure Animation Children Comedy Fantasy  
1                   Adventure Children Fantasy  
2                               Comedy Romance  
3                         Comedy Drama Romance  
4                                       Comedy  


In [90]:
movies_cb = movies[['movieId', 'title', 'genres']].copy()
movies_cb.head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action Crime Thriller
6,7,Sabrina (1995),Comedy Romance
7,8,Tom and Huck (1995),Adventure Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action Adventure Thriller


In [91]:
tfidf = TfidfVectorizer(stop_words='english')
movies_cb['content'] = (
    movies_cb['title']
    + ' '
    + movies_cb['genres']) 

tfidf_matrix = tfidf.fit_transform(
    movies_cb['content'])

print(tfidf_matrix.shape)

(9742, 9060)


In [92]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [93]:
def preprocess_title(title):
    title = re.sub(r'[.,]', '', title)
    title = title.strip().lower()
    return title

movies_cb['processed_title'] = movies_cb['title'].apply(preprocess_title)

In [94]:
from difflib import get_close_matches

def content_based_recommend(movie_title, top_n=10):

    movie_title = preprocess_title(movie_title)

    titles = movies_cb['processed_title'].tolist()

    matches = get_close_matches(movie_title, titles, n=1, cutoff=0.5)

    if not matches:
        print("Movie not found")
        return

    matched_title = matches[0]

    idx = movies_cb[movies_cb['processed_title'] == matched_title].index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
 
    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True )

    sim_scores = sim_scores[1:top_n+1] 

    movie_indices = [i[0] for i in sim_scores]

    results = movies_cb[['movieId','title', 'genres']].iloc[movie_indices].copy()

    results['similarity_score'] = [i[1] for i in sim_scores]

    return results

In [95]:
recommendations = content_based_recommend("Toy Story (1995)")

if recommendations is not None:

    print("\nRecommended Movies:\n")

    for i, row in recommendations.iterrows():

        print(f"Movie: {row['title']}")
        print(f"Genres: {row['genres']}")
        print(f"Similarity Score: {row['similarity_score']:.3f}")
        print("-" * 50)


Recommended Movies:

Movie: Toy Story 2 (1999)
Genres: Adventure Animation Children Comedy Fantasy
Similarity Score: 0.880
--------------------------------------------------
Movie: Toy Story 3 (2010)
Genres: Adventure Animation Children Comedy Fantasy IMAX
Similarity Score: 0.821
--------------------------------------------------
Movie: Toy, The (1982)
Genres: Comedy
Similarity Score: 0.538
--------------------------------------------------
Movie: We're Back! A Dinosaur's Story (1993)
Genres: Adventure Animation Children Fantasy
Similarity Score: 0.456
--------------------------------------------------
Movie: Now and Then (1995)
Genres: Children Drama
Similarity Score: 0.422
--------------------------------------------------
Movie: Toy Soldiers (1991)
Genres: Action Drama
Similarity Score: 0.401
--------------------------------------------------
Movie: NeverEnding Story, The (1984)
Genres: Adventure Children Fantasy
Similarity Score: 0.386
---------------------------------------------

In [96]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

reader = Reader(rating_scale=(0.5, 5)) 

data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader )

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train model
svd_model = SVD() 
svd_model.fit(trainset)

# Evaluate model
predictions = svd_model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

def collaborative_recommend(user_id, top_n=10):
    all_movie_ids = movies['movieId'].unique()

    rated_movies = set(ratings[ratings['userId'] == user_id]['movieId'].values )
    unseen_movies = [
        movie_id for movie_id in all_movie_ids
        if movie_id not in rated_movies ] 

    predict_list = []

    for movie_id in unseen_movies: 
        pred_rating = svd_model.predict( user_id, movie_id).est
        pred_rating = max(0.5, min(5.0, pred_rating))
        predict_list.append((movie_id, pred_rating))

    predict_list.sort(key=lambda x: x[1], reverse=True)

    top_recommendations = predict_list[:top_n]

    recommended_movies = []

    for movie_id, score in top_recommendations:
        title = movies.loc[movies['movieId'] == movie_id, 'title'].values[0]
        recommended_movies.append((title, round(score, 2)))

    return recommended_movies

recommendations = collaborative_recommend(user_id=1)

print("\nCollaborative Filtering Recommendations:\n")

for movie, score in recommendations:
    print(f"Movie: {movie}")
    print(f"Predicted Rating: {score:.2f}")
    print("-" * 50)

RMSE: 0.8798
MAE:  0.6764

Collaborative Filtering Recommendations:

Movie: Wallace & Gromit: The Best of Aardman Animation (1996)
Predicted Rating: 5.00
--------------------------------------------------
Movie: Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
Predicted Rating: 5.00
--------------------------------------------------
Movie: My Fair Lady (1964)
Predicted Rating: 5.00
--------------------------------------------------
Movie: Lawrence of Arabia (1962)
Predicted Rating: 5.00
--------------------------------------------------
Movie: Amadeus (1984)
Predicted Rating: 5.00
--------------------------------------------------
Movie: Bridge on the River Kwai, The (1957)
Predicted Rating: 5.00
--------------------------------------------------
Movie: Cool Hand Luke (1967)
Predicted Rating: 5.00
--------------------------------------------------
Movie: High Noon (1952)
Predicted Rating: 5.00
--------------------------------------------------
Movie: Sweet He

In [97]:
def hybrid_recommend(user_id, movie_title, top_n=10, w_content=0.5, w_collab=0.5):

    try:
        # Get content-based recommendations
        content_recs = content_based_recommend(movie_title, top_n=50)

        # Handle case: movie not found
        if content_recs is None or len(content_recs) == 0:
            print("No content-based recommendations found.")
            return []

        results = []

        for _, row in content_recs.iterrows():

            title = row['title']
            movie_id = row['movieId']

            # Content score
            content_score = row['similarity_score']
            content_score_norm = content_score
            #cosine similarity already in [0,1] check bs

            # Collaborative score
            try:
                collab_score = svd_model.predict(user_id, movie_id).est
            except:
                collab_score = 0
                # reason: fallback if prediction fails

            collab_score_norm = collab_score / 5.0
            #normalize SVD output to [0,1]

            final_score = (
                w_content * content_score_norm +
                w_collab * collab_score_norm
            )

            results.append((title, final_score))

        # sort results
        results.sort(key=lambda x: x[1], reverse=True)

        return results[:top_n]

    except Exception as e:
        print("Error in hybrid recommendation:", e)
        return []

In [98]:
hybrid_recommendations = hybrid_recommend(
    user_id=1,
    movie_title="Toy Story (1995)" )

In [99]:
print("\nHybrid Recommendations:\n")

for movie, score in hybrid_recommendations:
    print(f"Movie: {movie}")
    print(f"Final Score: {score:.3f}")
    print("-" * 50)


Hybrid Recommendations:

Movie: Toy Story 2 (1999)
Final Score: 0.913
--------------------------------------------------
Movie: Toy Story 3 (2010)
Final Score: 0.875
--------------------------------------------------
Movie: Toy, The (1982)
Final Score: 0.652
--------------------------------------------------
Movie: We're Back! A Dinosaur's Story (1993)
Final Score: 0.645
--------------------------------------------------
Movie: Up (2009)
Final Score: 0.636
--------------------------------------------------
Movie: Monsters, Inc. (2001)
Final Score: 0.633
--------------------------------------------------
Movie: Now and Then (1995)
Final Score: 0.628
--------------------------------------------------
Movie: Christmas Story, A (1983)
Final Score: 0.616
--------------------------------------------------
Movie: Shrek (2001)
Final Score: 0.607
--------------------------------------------------
Movie: The Lego Movie (2014)
Final Score: 0.604
--------------------------------------------------

In [100]:
weight_experiments = [
    (0.5, 0.5),
    (0.7, 0.3),
    (0.3, 0.7),
    (0.6, 0.4),
    (0.4, 0.6)
]

print("\nWeight Tuning Experiments:\n")

for w_content, w_collab in weight_experiments:

    recommendations = hybrid_recommend(
        user_id=1,
        movie_title="Toy Story (1995)",
        top_n=5,
        w_content=w_content,
        w_collab=w_collab
    )

    avg_score = np.mean([score for _, score in recommendations])

    print(f"Weights -> Content: {w_content}, Collaborative: {w_collab}")

    print(f"Average Score: {avg_score:.3f}")

    print("-" * 60)


Weight Tuning Experiments:

Weights -> Content: 0.5, Collaborative: 0.5
Average Score: 0.744
------------------------------------------------------------
Weights -> Content: 0.7, Collaborative: 0.3
Average Score: 0.695
------------------------------------------------------------
Weights -> Content: 0.3, Collaborative: 0.7
Average Score: 0.812
------------------------------------------------------------
Weights -> Content: 0.6, Collaborative: 0.4
Average Score: 0.719
------------------------------------------------------------
Weights -> Content: 0.4, Collaborative: 0.6
Average Score: 0.774
------------------------------------------------------------


In [101]:
def precision_recall_collaborative(predictions, k=10, threshold=3.5):

    user_predictions = {}
    for prediction in predictions:
        user_id = prediction.uid
        true_rating = prediction.r_ui 
        est_rating = prediction.est   
        
        if user_id not in user_predictions:
            user_predictions[user_id] = []
            
        user_predictions[user_id].append((est_rating, true_rating))
        
    total_precisions = []
    total_recalls = []
    
    for user_id, ratings in user_predictions.items():
        ratings.sort(key=lambda x: x[0], reverse=True)
        
        top_k_movies = ratings[:k]
        
        actually_liked = 0      
        recommended = 0         
        recommended_and_liked = 0 
        
        for est, true_r in ratings:
            if true_r >= threshold:
                actually_liked += 1
                
        for est, true_r in top_k_movies:
            if est >= threshold:
                recommended += 1
            if est >= threshold and true_r >= threshold:
                recommended_and_liked += 1
                
        if recommended != 0:
            user_precision = recommended_and_liked / recommended
        else:
            user_precision = 0
            
        if actually_liked != 0:
            user_recall = recommended_and_liked / actually_liked
        else:
            user_recall = 0
            
        total_precisions.append(user_precision)
        total_recalls.append(user_recall)
        
    mean_precision = sum(total_precisions) / len(total_precisions)
    mean_recall = sum(total_recalls) / len(total_recalls)
    
    return mean_precision, mean_recall

mean_precision, mean_recall = precision_recall_collaborative(predictions, k=10, threshold=3.5)

if (mean_precision + mean_recall) != 0:
    f1_score = (2 * mean_precision * mean_recall) / (mean_precision + mean_recall)
else:
    f1_score = 0

print(f"Precision: {mean_precision:.4f}")
print(f"Recall:  {mean_recall:.4f}")
print(f"F1-Score: {f1_score:.4f}")

Precision: 0.7380
Recall:  0.5114
F1-Score: 0.6042


In [102]:
def evaluate_content_based(top_n=10, threshold=3.5):

    precisions = []
    recalls = []

    users = ratings['userId'].unique()[:50]

    for user_id in users:

        user_movies = ratings[
            (ratings['userId'] == user_id) &
            (ratings['rating'] >= threshold)
        ]

        if len(user_movies) == 0:
            continue

        liked_movie = user_movies.iloc[0]['movieId']

        liked_title = movies[movies['movieId'] == liked_movie]['title'].values[0]

        recommendations = content_based_recommend(
            liked_title,
            top_n=top_n
        )

        if recommendations is None:
            continue

        recommended_ids = recommendations['movieId'].values

        relevant_movies = set(user_movies['movieId'].values)

        recommended_relevant = len(set(recommended_ids) & relevant_movies)

        precision = recommended_relevant / top_n

        recall = recommended_relevant / len(relevant_movies)

        precisions.append(precision)
        recalls.append(recall)

    mean_precision = np.mean(precisions)
    mean_recall = np.mean(recalls)

    if mean_precision + mean_recall != 0:
        f1 = ( 2 * mean_precision * mean_recall ) / (mean_precision + mean_recall)
    else:
        f1 = 0

    return mean_precision, mean_recall, f1

content_precision, content_recall, content_f1 = evaluate_content_based()

print("Content-Based Evaluation")

print(f"Precision: {content_precision:.4f}")
print(f"Recall: {content_recall:.4f}")
print(f"F1-Score: {content_f1:.4f}")

Content-Based Evaluation
Precision: 0.0380
Recall: 0.0031
F1-Score: 0.0057


In [103]:
def evaluate_hybrid(top_n=10, threshold=3.5):

    precisions = []
    recalls = []

    users = ratings['userId'].unique()[:50]

    for user_id in users:

        user_movies = ratings[
            (ratings['userId'] == user_id) &
            (ratings['rating'] >= threshold)
        ]

        if len(user_movies) == 0:
            continue

        liked_movie = user_movies.iloc[0]['movieId']

        liked_title = movies[movies['movieId'] == liked_movie]['title'].values[0]

        recommendations = hybrid_recommend(
            user_id=user_id,
            movie_title=liked_title,
            top_n=top_n
        )

        recommended_titles = [ movie for movie, score in recommendations ]

        relevant_titles = movies[movies['movieId'].isin(user_movies['movieId'])]['title'].values

        recommended_relevant = len( set(recommended_titles) & set(relevant_titles) )

        precision = recommended_relevant / top_n

        recall = recommended_relevant / len(relevant_titles)

        precisions.append(precision)
        recalls.append(recall)

    mean_precision = np.mean(precisions)
    mean_recall = np.mean(recalls)

    if mean_precision + mean_recall != 0:
        f1 = (
            2 * mean_precision * mean_recall
        ) / (mean_precision + mean_recall)
    else:
        f1 = 0

    return mean_precision, mean_recall, f1

hybrid_precision, hybrid_recall, hybrid_f1 = evaluate_hybrid()

print("Hybrid Model Evaluation")

print(f"Precision: {hybrid_precision:.4f}")
print(f"Recall: {hybrid_recall:.4f}")
print(f"F1-Score: {hybrid_f1:.4f}")

Hybrid Model Evaluation
Precision: 0.0720
Recall: 0.0057
F1-Score: 0.0105


In [ ]:
def hybrid_predict(user_id, movie_id, movie_title,
                   w_content=0.5,
                   w_collab=0.5):

    # collaborative prediction
    collab_score = svd_model.predict(user_id, movie_id).est

    # content similarity
    processed_title = preprocess_title(movie_title)

    titles = movies_cb['processed_title'].tolist()

    matches = get_close_matches(
        processed_title,
        titles,
        n=1,
        cutoff=0.5
    )

    if matches:

        matched_title = matches[0]

        idx = movies_cb[movies_cb['processed_title'] == matched_title].index[0]

        movie_idx = movies_cb[movies_cb['movieId'] == movie_id].index

        if len(movie_idx) > 0:

            similarity = cosine_sim[idx][movie_idx[0]]

        else:
            similarity = 0

    else:
        similarity = 0

    # normalize collaborative
    collab_norm = collab_score / 5.0

    # hybrid score
    hybrid_score = (
        w_content * similarity +
        w_collab * collab_norm
    )

    # convert back to rating scale
    predicted_rating = hybrid_score * 5

    return predicted_rating

In [105]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
import math

true_ratings = []
predicted_ratings = []

for row in ratings.sample(1000).itertuples():

    user_id = row.userId
    movie_id = row.movieId
    true_rating = row.rating

    movie_title = movies[movies['movieId'] == movie_id]['title'].values[0]

    pred_rating = hybrid_predict(
        user_id,
        movie_id,
        movie_title
    )

    true_ratings.append(true_rating)
    predicted_ratings.append(pred_rating)

# RMSE
rmse_hybrid = math.sqrt(
    mean_squared_error(
        true_ratings,
        predicted_ratings
    )
)

# MAE
mae_hybrid = mean_absolute_error(
    true_ratings,
    predicted_ratings
)

print("Hybrid Model Evaluation")

print(f"RMSE: {rmse_hybrid:.4f}")
print(f"MAE: {mae_hybrid:.4f}")


Hybrid Model Evaluation
RMSE: 1.1348
MAE: 0.8906


In [106]:
comparison_df = pd.DataFrame({

    'Model': [
        'Content-Based',
        'Collaborative Filtering',
        'Hybrid Model'
    ],

    'RMSE': [
        'N/A',
        rmse,
        rmse_hybrid
    ],

    'MAE': [
        'N/A',
        mae,
        mae_hybrid
    ],

    'Precision': [
        content_precision,
        mean_precision,
        hybrid_precision
    ],

    'Recall': [
        content_recall,
        mean_recall,
        hybrid_recall
    ],

    'F1-Score': [
        content_f1,
        f1_score,
        hybrid_f1
    ]
})

print(" Model Comparison:\n")
print(comparison_df)

 Model Comparison:

                     Model      RMSE       MAE  Precision    Recall  F1-Score
0            Content-Based       N/A       N/A   0.038000  0.003106  0.005743
1  Collaborative Filtering  0.879756  0.676434   0.738009  0.511403  0.604156
2             Hybrid Model  1.134797  0.890554   0.072000  0.005683  0.010535
